# Change the following lines to adapt to each assignment

This notebook mirrors the style of the original autograder, but is specialized for GW08 (Table 3 two-column replication).

In [23]:
import json
import os
import nbformat
from dotenv import load_dotenv
from openai import OpenAI

In [24]:
# Assignment + file paths
assignment_name = "GW08 Table 3 (two columns)"

assignment_notebook_path = "GW08_table3_two_cols.ipynb"
submission_notebook_path = "submission_sample/GW08_table3_two_cols_student_submission_medium_wrong.ipynb"
rubric_path = "rubric_gw.txt"

# Output folder (for saving student results)
results_folder = "results/submission_student_a"

# Create results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

grading_json_path = f"{results_folder}/grading_result.json"
feedback_md_path = f"{results_folder}/student_feedback.md"
feedback_pdf_path = f"{results_folder}/student_feedback.pdf"

# Helper functions

In [25]:
def load_notebook(path):
    with open(path, "r", encoding="utf-8") as f:
        return nbformat.read(f, as_version=4)


def _extract_output_text(outputs):
    out_parts = []

    for out in outputs or []:
        out_type = out.get("output_type", "")

        if out_type == "stream":
            text = out.get("text", "")
            if isinstance(text, list):
                text = "".join(text)
            out_parts.append(str(text))

        elif out_type in ["execute_result", "display_data"]:
            data = out.get("data", {})
            plain = data.get("text/plain", "")
            if isinstance(plain, list):
                plain = "".join(plain)
            if plain:
                out_parts.append(str(plain))

        elif out_type == "error":
            ename = out.get("ename", "Error")
            evalue = out.get("evalue", "")
            tb = out.get("traceback", [])
            tb_text = "\n".join(tb) if tb else ""
            out_parts.append(f"{ename}: {evalue}\n{tb_text}".strip())

    return "\n".join([x for x in out_parts if x]).strip()


def extract_notebook_content(path):
    nb = load_notebook(path)

    markdown_cells = []
    code_cells = []
    outputs = []
    merged_text = []

    for i, cell in enumerate(nb.cells, start=1):
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)

        if cell.get("cell_type") == "markdown":
            markdown_cells.append(src)
            merged_text.append(f"[Markdown Cell {i}]\n{src}")

        if cell.get("cell_type") == "code":
            code_cells.append(src)
            merged_text.append(f"[Code Cell {i}]\n{src}")

            out_text = _extract_output_text(cell.get("outputs", []))
            if out_text:
                outputs.append(out_text)
                merged_text.append(f"[Output Cell {i}]\n{out_text}")

    return {
        "path": path,
        "num_cells": len(nb.cells),
        "markdown_cells": markdown_cells,
        "code_cells": code_cells,
        "outputs": outputs,
        "prompt_text": "\n\n".join(merged_text)
    }


def load_rubric(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [26]:
REQUIRED_OUTPUT_SCHEMA = {
    "total_score": 0,
    "max_score": 100,
    "breakdown": [
        {
            "criterion": "string",
            "score": 0,
            "max_score": 0,
            "reason": "string"
        }
    ],
    "final_feedback": "string",
    "flags_for_manual_review": ["string"]
}


def build_grading_prompt(assignment_text, submission_text, rubric):
    grading_rules = """You are grading a finance replication assignment (GW08).

Assignment scope:
- Replicate Goyal and Welch Table 3 with two columns: IS_R2_head and OOS_R2_head.
- Do NOT grade tangency portfolios, Sharpe ratios, equal-weight portfolios, or portfolio optimization.

Important grading rules:
- Use the rubric exactly as provided.
- Give partial credit for correct methodology.
- Accept small numerical differences.
- Penalize look-ahead bias in rolling 20-year (240-month) out-of-sample forecasting.
- Final output should contain IS_R2_head and OOS_R2_head.
- If notebook is empty or unrelated, assign a low score and explain why.
- If uncertain, add concise notes to flags_for_manual_review.

Return valid JSON only (no markdown fences) with this exact schema:"""

    prompt = f"""
{grading_rules}
{json.dumps(REQUIRED_OUTPUT_SCHEMA, indent=2)}

Rubric JSON:
{json.dumps(rubric, indent=2)}

Original assignment notebook content:
{assignment_text}

Student submission notebook content:
{submission_text}

Grade now and return only JSON.
"""
    return prompt

In [27]:
# LLM setup from .env
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "").strip()
api_key = REDACTED    groq_key
    or os.getenv("OPENAI_API_KEY", "").strip()
    or os.getenv("GROK_API_KEY", "").strip()
)
model_name = os.getenv("MODEL_NAME", "").strip()

base_url = (
    os.getenv("GROQ_BASE_URL", "").strip()
    or os.getenv("OPENAI_BASE_URL", "").strip()
    or os.getenv("GROK_BASE_URL", "").strip()
)

if groq_key:
    base_url = "https://api.groq.com/openai/v1"
elif not base_url:
    base_url = "https://api.openai.com/v1"

if not api_key:REDACTED    raise ValueError("Missing API key. Set GROQ_API_KEY or another OpenAI-compatible API key in .env.")
if not model_name:
    raise ValueError("Missing MODEL_NAME in .env.")

client_kwargs = {"api_key": api_key}
if base_url:
    client_kwargs["base_url"] = base_url

client = OpenAI(**client_kwargs)
print(f"Client ready. Model: {model_name}")

Client ready. Model: llama-3.3-70b-versatile


In [28]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.units import inch


def call_llm(prompt):
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": "You are a strict grading assistant. Return valid JSON only, no extra text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    
    content = response.choices[0].message.content
    if content is None:
        raise ValueError("LLM returned empty content.")
    return content


def parse_llm_json(response_text):
    text = response_text.strip()

    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    result = json.loads(text)

    required_keys = [
        "total_score",
        "max_score",
        "breakdown",
        "final_feedback",
        "flags_for_manual_review"
    ]

    for k in required_keys:
        if k not in result:
            raise ValueError(f"Missing required key in grading JSON: {k}")

    return result


def save_json(result, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)


def save_markdown_feedback(result, output_path):
    lines = []

    lines.append("# GW08 Autograder Feedback")
    lines.append("")
    lines.append(f"## Score: {result.get('total_score', 0)}/{result.get('max_score', 100)}")
    lines.append("")

    lines.append("## Rubric Breakdown")
    lines.append("")
    breakdown = result.get("breakdown", [])

    if breakdown:
        for item in breakdown:
            criterion = item.get("criterion", "Unknown criterion")
            score = item.get("score", 0)
            max_score = item.get("max_score", 0)
            reason = item.get("reason", "")
            lines.append(f"- **{criterion}**: {score}/{max_score}")
            lines.append(f"  - Reason: {reason}")
    else:
        lines.append("- No rubric breakdown returned.")

    lines.append("")
    lines.append("## Final Feedback")
    lines.append("")
    lines.append(result.get("final_feedback", "No final feedback provided."))
    lines.append("")

    lines.append("## Flags For Manual Review")
    lines.append("")
    flags = result.get("flags_for_manual_review", [])
    if flags:
        for flag in flags:
            lines.append(f"- {flag}")
    else:
        lines.append("- None")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines).rstrip() + "\n")


def markdown_to_pdf(md_content, pdf_output_path):
    """Convert markdown feedback to PDF using reportlab."""
    doc = SimpleDocTemplate(pdf_output_path, pagesize=letter,
                           topMargin=0.5*inch, bottomMargin=0.5*inch)
    styles = getSampleStyleSheet()
    
    # Custom styles
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=18,
        textColor='black',
        spaceAfter=12,
        alignment=TA_CENTER
    )
    heading_style = ParagraphStyle(
        'CustomHeading',
        parent=styles['Heading2'],
        fontSize=14,
        textColor='black',
        spaceAfter=8,
        spaceBefore=8
    )
    
    story = []
    
    # Parse markdown and build PDF elements
    lines = md_content.split('\n')
    for i, line in enumerate(lines):
        if line.startswith('# '):
            story.append(Paragraph(line[2:], title_style))
            story.append(Spacer(1, 0.2*inch))
        elif line.startswith('## '):
            story.append(Paragraph(line[3:], heading_style))
            story.append(Spacer(1, 0.1*inch))
        elif line.startswith('- '):
            story.append(Paragraph('• ' + line[2:], styles['BodyText']))
            story.append(Spacer(1, 0.05*inch))
        elif line.strip():
            story.append(Paragraph(line, styles['BodyText']))
            story.append(Spacer(1, 0.1*inch))
    
    # Build PDF
    doc.build(story)
    print(f"✓ PDF generated: {pdf_output_path}")

ModuleNotFoundError: No module named 'reportlab'

# Run grading

In [ ]:
assignment_data = extract_notebook_content(assignment_notebook_path)
submission_data = extract_notebook_content(submission_notebook_path)
rubric = load_rubric(rubric_path)

prompt = build_grading_prompt(
    assignment_text=assignment_data["prompt_text"],
    submission_text=submission_data["prompt_text"],
    rubric=rubric
)

llm_response_text = call_llm(prompt)
result = parse_llm_json(llm_response_text)

# Save outputs
save_json(result, grading_json_path)
save_markdown_feedback(result, feedback_md_path)

# Generate PDF from markdown feedback
with open(feedback_md_path, "r", encoding="utf-8") as f:
    md_content = f.read()
markdown_to_pdf(md_content, feedback_pdf_path)

print("Autograding complete.")
print(f"Score: {result['total_score']}/{result['max_score']}")
print(f"Saved:")
print(f"  - JSON: {grading_json_path}")
print(f"  - Markdown: {feedback_md_path}")
print(f"  - PDF: {feedback_pdf_path}")

Autograding complete.
Score: 60/100
Saved: grading_result.json
Saved: student_feedback.md


In [ ]:
# Optional quick preview
result

{'total_score': 60,
 'max_score': 100,
 'breakdown': [{'criterion': 'Attempt and notebook completeness',
   'score': 8,
   'max_score': 10,
   'reason': 'The submission is non-empty and includes meaningful code toward the Table 3 replication, but has several mistakes.'},
  {'criterion': 'Data loading',
   'score': 10,
   'max_score': 10,
   'reason': 'Correctly loads the monthly Goyal-Welch data file and handles separator/decimal formatting.'},
  {'criterion': 'Date parsing and sample filtering',
   'score': 8,
   'max_score': 10,
   'reason': 'Correctly converts yyyymm to dates and sets a date index, but uses slightly wrong start dates for some variables.'},
  {'criterion': 'Predictor construction',
   'score': 12,
   'max_score': 15,
   'reason': 'Mostly correctly constructs key predictors, but may have minor issues with specific variables.'},
  {'criterion': 'Lag handling',
   'score': 6,
   'max_score': 10,
   'reason': 'Incorrectly uses a 1-month lag for all predictors, including 